In [3]:
!pip install sentence-transformers chromadb groq pandas -q
print("Installed successfully!")

Installed successfully!


In [8]:
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer
from groq import Groq
import os

In [13]:
GROQ_API_KEY = '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'
os.environ["GROQ_API_KEY"] = GROQ_API_KEY
groq_client = Groq(api_key=GROQ_API_KEY)
print("groq API Client Initialized")

groq API Client Initialized


In [19]:
df = pd.read_csv('college_notes.csv')
print("Shape of dataset: ", df.shape)
print("\nColumn names: ", df.columns.tolist())
print("\nFirst 3 rows:")
df.head(3)

Shape of dataset:  (15, 4)

Column names:  ['note_id', 'subject', 'topic', 'content']

First 3 rows:


,note_id,subject,topic,content
0,N001,Data Engineering,ETL Pipelines,ETL stands for Extract Transform Load. It is t...
1,N002,Data Engineering,SQL Databases,A database is an organized collection of data ...
2,N003,Data Engineering,Data Cleaning,Data cleaning involves fixing or removing inco...


In [34]:
print("Subjects in this dataset:")
print(df['subject'].value_counts())
print("\nSample of topics:")
print(df[['note_id','subject','topic','topic']].head(3).to_string(index = False))
print("\nLength of content (number of characters) for each note:")
df['content_length'] = df['content'].apply(len)
print(df[['topic', 'content_length']].to_string(index = False))

Subjects in this dataset:
subject
Machine Learning      4
Data Engineering      3
Generative AI         3
Python Programming    2
Name: count, dtype: int64

Sample of topics:
note_id          subject                    topic                    topic
   N003 Data Engineering            Data Cleaning            Data Cleaning
   N004 Data Engineering APIs and Data Collection APIs and Data Collection
   N005 Data Engineering     Big Data and PySpark     Big Data and PySpark

Length of content (number of characters) for each note:
                         topic  content_length
                 Data Cleaning             210
      APIs and Data Collection             224
          Big Data and PySpark             242
           Supervised Learning             255
           Feature Engineering             236
                Decision Trees             227
                 Random Forest             238
         Large Language Models             226
            Prompt Engineering             27

In [38]:
df = df.drop(0)
documents = df['content'].tolist()
ids = [f"note_{row['note_id']}" for row in df.to_dict('records')]
metadatas = [
    {"subject": row['subject'], "topic": row['topic']}
    for row in df.to_dict('records')
]

print(f"Total chunks prepared : {len(documents)}")
print(f"First document id: {ids[0]}")
print(f"First document metadata: {metadatas[0]}")
print(f"First 100 characters of doc: {documents[0][:100]}...")

Total chunks prepared : 14
First document id: note_N002
First document metadata: {'subject': 'Data Engineering', 'topic': 'SQL Databases'}
First 100 characters of doc: A database is an organized collection of data stored electronically. SQL or Structured Query Languag...


In [39]:
print("Loading embedding model...")

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
print("\nEmbedding model loaded successfully!")
test_embedding = embedding_model.encode("This is a test sentence")
print(f"Test embedding shape: {test_embedding.shape}")
print(f"First 5 values of test embedding: {test_embedding[:5]}")

Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


Embedding model loaded successfully!
Test embedding shape: (384,)
First 5 values of test embedding: [0.07155243 0.06848023 0.00660337 0.10176966 0.01112225]


In [41]:
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(name = "college_notes_rag")
print("ChromaDB client created")
print(f"Collection name: college_notes_rag")
print(f"Documents in collection so far: {collection.count()}")

ChromaDB client created
Collection name: college_notes_rag
Documents in collection so far: 0


In [44]:
print("Generating embeddings for all 15 notes...")
embeddings = embedding_model.encode(documents, show_progress_bar=True)
print(f"\nEmbedding matrix shape: {embeddings.shape}")
embeddings_list = embeddings.tolist()
collection.add(
    documents = documents,
    metadatas = metadatas,
    ids = ids,
    embeddings = embeddings_list
)
print(f"\nDocuments added successfully!")
print(f"Total documents in collection: {collection.count()}")

Generating embeddings for all 15 notes...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Embedding matrix shape: (14, 384)

Documents added successfully!
Total documents in collection: 14


In [57]:
def retrieve_relevant_chunks(question, top_k = 3):
  """
  Given a user question, retrieve the most relevant document chunks from chromaDB.

  Parameters:
  question(str) : The user's question as a text string
  top_k(int) : Howmnay top results to return (default : 3)

  Returns:
  A dictionary containing retrieved documents, distances and metadata
  """

  question_embedding = embedding_model.encode(question).tolist()
  results = collection.query(
      query_embeddings = [question_embedding],
      n_results = top_k
  )
  return results
print("Retrieval function defined successfully!")

Retrieval function defined successfully!


In [58]:
test_question = "What is Python Programming and its real world applications?"
print(f"Test question: {test_question}")
print()
results = retrieve_relevant_chunks(test_question, top_k = 4)
print("\nTop 4 retrieved chunks:")
print()

for i, (doc, dist, meta) in enumerate(zip(
  results['documents'][0],
  results['distances'][0],
  results['metadatas'][0]
)):
  print(f"\nResult: {i+1}")
  print(f"  Subject: {meta['subject']}")
  print(f"  Topic: {meta['topic']}")
  print(f"  Distance: {dist}")
  print(f"  Content: {doc[:120]}...")

Test question: What is Python Programming and its real world applications?


Top 4 retrieved chunks:


Result: 1
  Subject: Python Programming
  Topic: Pandas Library
  Distance: 1.1065406799316406
  Content: Pandas is a Python library used for data manipulation and analysis. It provides the DataFrame data structure which is li...

Result: 2
  Subject: Data Engineering
  Topic: APIs and Data Collection
  Distance: 1.1180229187011719
  Content: An API or Application Programming Interface allows two software applications to talk to each other. In data engineering ...

Result: 3
  Subject: Python Programming
  Topic: Data Visualization
  Distance: 1.146761417388916
  Content: Data visualization is the process of representing data as charts graphs and visual formats. Python libraries like Matplo...

Result: 4
  Subject: Data Engineering
  Topic: Big Data and PySpark
  Distance: 1.150917649269104
  Content: Big Data refers to extremely large datasets that cannot be processed by traditional 

In [ ]:
def build_context_from_results(results):
  """
  Format chromaDB retrieval results into a readable context string.

  Parameters:
  results: The output from collection.query() - a dictionary.

  Returns:
  context_str (str): A formatted String of all retrieved document chunks
  """

  context_parts = []
  for i, (doc, meta) in enumerate(zip(
      results['documents'][0],
      results['metadatas'][0]
  )):

      chunk_text = f"[Source {i+1}: {meta['subject']} - {meta['topic']}]\n{doc}"

In [59]:
def generate_rag_answer(question, context):
    """
    Send the retrieved context and question to the Groq LLM for answer generation.
    """
    system_prompt = """You are a helpful academic assistant for engineering students.
    You will be given context retrieved from a college knowledge base and student's question.
    RULES:
    1. Answer only using the information provided in the context below.
    2. If the answer is not found in the context, say exactly:
    "I dont have enough information in my knowledge base to answer my question"
    3. Do not use your general training knowledge
    4. Keep answers clear, accurate and beginner-friendly
    5. Mention which source the information came from when possible.
    """

    user_prompt = f"""Context from knowledge base:
{context}
---
Student's Question: {question}

Please answer the question based only on the context provided above."""

    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.1,
        max_tokens=500
    )
    answer = response.choices[0].message.content
    return answer
print("RAG generation function defined!")

RAG generation function defined!


In [ ]:
def ask_college_assistant(question, top_k = 3, verbose = True):
  """
  Complete RAG pipeline : Given a question, retrieve relevant context and generate an answer.

  Parameters:
  question (str): The user's question
  top_k (int) : Number of chunks to retrieve (default : 3)
  verbose (bool) : Whether to print intermediate steps (default : True)

  Returns:
  answer (str) : The final generated answer
  """

  if verbose:
    print(f"Questio: {question}")
    print()
    print("Step 1: Retrieving relevant documents...")

  results = retrieve_relevant_chunks(question, top_k = top_k)